# A.5 答錯是沒找到還是沒用好？


問三家書店的新地址，系統只答對一家。這能告訴我們最終表現，卻還不知道該修哪一段：公告根本沒找進來，還是公告已在提示裡，模型卻讀錯地址？前置是[檢索流程](https://birdhackor.github.io/tiny-perceptron-vlm/A.3.html)與[證據支持](https://birdhackor.github.io/tiny-perceptron-vlm/A.4.html)。把中途資料留下來，才可能分清找資料與用資料的錯誤。

先為每題人工指定應找到的正確來源，再記錄取回的文件編號與實際答案。若正確來源出現在取回清單，本節將hit記True；若答案符合固定真值，correct記True。第一題兩者皆True，第二題找到公告但答錯，第三題沒找到也答錯。這三筆是人造診斷例，不是模型評測成績，數字只用來展示兩種計數的差別。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"question": "甲店地址", "hit": True, "correct": True},
    {"question": "乙店地址", "hit": True, "correct": False},
    {"question": "丙店地址", "hit": False, "correct": False},
]
count = len(examples)
recall = sum(e["hit"] for e in examples) / count
accuracy = sum(e["correct"] for e in examples) / count
print("有正確來源", recall)
print("答案正確", accuracy)
for e in examples:
    if not e["correct"]:
        print(e["question"], "已找到但答錯" if e["hit"] else "尚未找到所需來源")

`sum`把True當1、False當0，加總後除以三題。有正確來源是2/3，約0.6667；答案正確是1/3，約0.3333。後兩行依序指出乙店「已找到但答錯」、丙店「尚未找到所需來源」。前者應先檢查提示有沒有完整保留公告、地址是否被混淆；後者應先檢查文件是否存在、查詢與排序是否選到它。不要把兩種錯誤全算成模型不會回答。

這裡的來源命中比例可稱retrieval recall；本節每題只有一個指定來源，所以分母是題數。若一題需要多份文件，還要說清楚是「至少找到一份」還是「所需文件全部找到」，不能直接沿用同一含糊標籤。answer accuracy則是答案正確題數除以題數，需要固定答案判準，例如地址欄位完全吻合，或人工核對語意。

進一步可把正確公告直接提供給原本失敗的題目，固定模型與生成設定重新回答。若這時答對，表示原本的檢索階段很可能是瓶頸；若仍答錯，就繼續查資料放置、閱讀和回答。這種介入比只看總分更有資訊，但不是絕對因果證明：提示長度或其他文件也一起改變時，仍要記錄差異。即使沒有hit卻碰巧答對，也可能靠舊記憶或猜測，不能當作引用可靠。

練習只把第二題correct改True，hit全部不動。先預測來源比例仍2/3，答案比例升到2/3，錯誤清單只剩丙店，再執行核對。這個改動表示「用已找到的資料答對了」，不是檢索器找到更多公告；你應能依兩個數字說明改善發生在哪一段。
